In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")
    

Gemini API Key exists and begins AQ.Ab8RN


In [5]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I help you today?


In [6]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Help users understand their pre-provided medication schedule.
        Do not invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "What can you help me with?"
    }
]

In [7]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

print(response.choices[0].message.content)

I am your **AI Medication Assistant**. I am here to help you stay organized and informed about the medication schedule you have provided.

Specifically, I can help you with the following:

*   **Schedule Clarification:** I can explain how to read your medication regimen and help you understand when each dose is due based on the information you’ve provided.
*   **Dose Reminders:** I can help you organize your daily routine so you know exactly what to take and when.
*   **Terminology Support:** If you have questions about specific terms found on your prescription labels or instructions (such as "PRN," "with food," or "twice daily"), I can define them for you.
*   **Organization:** I can help you create a clear, easy-to-read table or list of your medications.

---

### ⚠️ Important Safety Guidelines:
*   **I do not provide medical advice:** I cannot diagnose conditions, recommend changes to your prescribed dosages, or suggest new medications. 
*   **Consult your professional:** Always tal

In [8]:
medications = [
    {
        "name": "Paracetamol",
        "dosage": "500 mg",
        "time": "8:00 AM",
        "instructions": "Take after breakfast"
    },
    {
        "name": "Metformin",
        "dosage": "500 mg",
        "time": "8:00 PM",
        "instructions": "Take after dinner"
    }
]

medication_context = str(medications)

In [9]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.

            Help users understand their pre-provided medication schedule.
            Only use the medication information provided to you.
            Never invent medication information or change a prescribed dosage.
            If the required information is not provided, say that you don't have that information.
            """
        },
        {
            "role": "user",
            "content": f"""
            Here is the user's medication information:

            {medication_context}

            What medicines does the user take in the morning?
            """
        }
    ]
)

print(response.choices[0].message.content)

Based on the information provided, you take Paracetamol (500 mg) at 8:00 AM, which should be taken after breakfast.


In [10]:
def get_medicines():
    return medications

print(get_medicines())

[{'name': 'Paracetamol', 'dosage': '500 mg', 'time': '8:00 AM', 'instructions': 'Take after breakfast'}, {'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}]


In [11]:
get_medicines_tool = {
    "type": "function",
    "function": {
        "name": "get_medicines",
        "description": "Get the user's complete medication list and schedule.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}

In [12]:
tools = [get_medicines_tool]

In [13]:
print(tools)

[{'type': 'function', 'function': {'name': 'get_medicines', 'description': "Get the user's complete medication list and schedule.", 'parameters': {'type': 'object', 'properties': {}, 'required': []}}}]


In [14]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.
            When the user asks about their medications,
            use the available tool to retrieve their medication information.
            """
        },
        {
            "role": "user",
            "content": "What medicines am I currently taking?"
        }
    ],
    tools=tools
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_2716644', function=Function(arguments='{}', name='get_medicines'), type='function', extra_content={'google': {'thought_signature': 'EnEKbwERTTIPz5VRZmbb4FaX0OkTK8e0/drjjoALlC13gxF5sXRpzv3ptZozW5p96Viz+u/lyxpznnS8Zz6aI6rLt25UWhUsfNrjXmBvo1BO6Hfeq5UCIMucTmW+3Baj2mosmvfjkksI/v8VaUOrUetOMw=='}})])


In [15]:
tool_call = response.choices[0].message.tool_calls[0]

print("Tool name:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

Tool name: get_medicines
Arguments: {}


In [16]:
tool_result = get_medicines()

print(tool_result)

[{'name': 'Paracetamol', 'dosage': '500 mg', 'time': '8:00 AM', 'instructions': 'Take after breakfast'}, {'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}]


In [17]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Help users understand their pre-provided medication schedule.
        Only use the medication information provided by the tool.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "What medicines am I currently taking?"
    },
    response.choices[0].message,
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(tool_result)
    }
]

In [18]:
final_response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

You are currently taking the following medications:

*   **Paracetamol (500 mg):** Take at 8:00 AM after breakfast.
*   **Metformin (500 mg):** Take at 8:00 PM after dinner.


In [19]:
available_tools = {
    "get_medicines": get_medicines
}

In [20]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
function_to_call = available_tools[function_name]

tool_result = function_to_call()

print(tool_result)

[{'name': 'Paracetamol', 'dosage': '500 mg', 'time': '8:00 AM', 'instructions': 'Take after breakfast'}, {'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}]


In [21]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.
            When the user asks about their medications,
            use the available tool to retrieve their medication information.
            """
        },
        {
            "role": "user",
            "content": "What medicines am I currently taking?"
        }
    ],
    tools=tools
)

In [22]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
function_to_call = available_tools[function_name]

tool_result = function_to_call()

print(tool_result)

[{'name': 'Paracetamol', 'dosage': '500 mg', 'time': '8:00 AM', 'instructions': 'Take after breakfast'}, {'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}]


In [23]:
def get_medicine_by_name(name):
    for medicine in medications:
        if medicine["name"].lower() == name.lower():
            return medicine
    return {"error": "Medicine not found"}

In [24]:
print(get_medicine_by_name("Metformin"))

{'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}


In [25]:
get_medicine_by_name_tool = {
    "type": "function",
    "function": {
        "name": "get_medicine_by_name",
        "description": "Get the details of a medicine by its name.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "The name of the medicine."
                }
            },
            "required": ["name"]
        }
    }
}

In [26]:
tools = [
    get_medicines_tool,
    get_medicine_by_name_tool
]

In [27]:
available_tools = {
    "get_medicines": get_medicines,
    "get_medicine_by_name": get_medicine_by_name
}

In [28]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {
            "role": "system",
            "content": """
            You are an AI Medication Assistant.
            When the user asks about a specific medicine,
            use the appropriate medication tool.
            """
        },
        {
            "role": "user",
            "content": "Tell me the details of Metformin."
        }
    ],
    tools=tools
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_242913', function=Function(arguments='{"name":"Metformin"}', name='get_medicine_by_name'), type='function', extra_content={'google': {'thought_signature': 'EnEKbwERTTIPDbTORSMBBUSoQ4T/uToOBqARCCO7yWO6/ggoC/vpscWhS3xS6KpPLsRmUTEYgTBEf3yReVF3AsEBVyMcyOeNAWpTRpfcIlog2r/nb7KpMTIkuuOOYyTwmzbbFlRdXSUR4YIlvvje03rrtQ=='}})])


In [29]:
tool_call = response.choices[0].message.tool_calls[0]

print("Function:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

Function: get_medicine_by_name
Arguments: {"name":"Metformin"}


In [31]:
import json

arguments = json.loads(tool_call.function.arguments)

print(arguments)

{'name': 'Metformin'}


In [32]:
tool_result = function_to_call(**arguments)

print(tool_result)

TypeError: get_medicines() got an unexpected keyword argument 'name'

In [33]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

function_to_call = available_tools[function_name]

print("Function name:", function_name)
print("Arguments:", arguments)
print("Function to call:", function_to_call.__name__)

Function name: get_medicine_by_name
Arguments: {'name': 'Metformin'}
Function to call: get_medicine_by_name


In [34]:
tool_result = function_to_call(**arguments)

print(tool_result)

{'name': 'Metformin', 'dosage': '500 mg', 'time': '8:00 PM', 'instructions': 'Take after dinner'}


In [35]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

function_to_call = available_tools[function_name]

tool_result = function_to_call(**arguments)

In [36]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Help users understand their pre-provided medication information.
        Only use information returned by the available tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "Tell me the details of Metformin."
    },
    response.choices[0].message,
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(tool_result)
    }
]

final_response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

Metformin is prescribed at a dosage of 500 mg, to be taken at 8:00 PM. Please make sure to take it after dinner.


In [37]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.
        Use the available tools when medication information is required.
        Only use information returned by the tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": "Tell me the details of Metformin."
    }
]

while True:

    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)

    for tool_call in message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function_to_call = available_tools[function_name]

        tool_result = function_to_call(**arguments)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

Metformin is prescribed as 500 mg, to be taken after dinner at 8:00 PM.


In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        You are an AI Medication Assistant.

        Use the available tools whenever medication information is required.
        Only use information returned by the tools.
        Never invent medication information or change prescribed dosages.
        """
    },
    {
        "role": "user",
        "content": """
        Give me the details of both Paracetamol and Metformin.
        """
    }
]

while True:

    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)

    for tool_call in message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function_to_call = available_tools[function_name]

        tool_result = function_to_call(**arguments)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

Here are the details for your medications:

*   **Paracetamol**: 500 mg, to be taken at 8:00 AM after breakfast.
*   **Metformin**: 500 mg, to be taken at 8:00 PM after dinner.
